In [ ]:

import numpy as np
import tifffile
from ome_types import to_xml
from ome_types.model import OME, Image, Pixels, Channel, TiffData

from pathlib import Path
import numpy as np
from bioio import BioImage
import bioio_ome_tiff
import tifffile
from ome_types import from_xml
import zarr

import numpy as np
import zarr
import os

from ome_zarr.io import parse_url
from ome_zarr.writer import write_image

## Best shot so far
- Read a 6D FILM data from .zarr file converted using plugin (napari-flim-phasor-plotter) (here creating a small syntehtic array)
- Convert to numpy (load all into memory...)
- swap time and flim axis to have time first
- merge time and flim axes to turn array into 5D (information about original axes will be given as an annotation as explained in the modulo dataset examples from omero)
- change axes order to start with xy (because ome_types only accepts them like that!)
- Write the 5D array as ome-tif with tiffile WORKS!
BUT writing the 5D array as ome-zarr does not work, since `write_image` from OME-Zarr asks time to come first...

In [ ]:

# Create some dummy image data (6D: C, H, T, Z, Y, X)
data = np.random.randint(0, 256, (3, 10, 2, 5, 256, 256), dtype=np.uint8)
data = np.moveaxis(data, [0, 1, 2, 3, 4, 5], [0, 2, 1, 3, 4, 5]) # CHTZYX -> CTHZYX (swap T and H to make T come first)
data = np.reshape(data, (3, 20, 5, 256, 256)) # C[TH]ZYX Merge T and H dimensions
# data = np.moveaxis(data, [0, 1, 2, 3, 4], [4, 3, 2, 1, 0]) # C[TH]ZYX -> XYZ[TH]C (used if wirting with tifffile)
# OR put time first
data = np.moveaxis(data, [0, 1, 2, 3, 4], [1, 0, 2, 3, 4]) # C[TH]ZYX -> [TH]CZYX


print(data.shape)

# Create OME metadata without the structured annotations
def create_ome_metadata(data_shape):
    # x_size, y_size, z_size, t_size, c_size = data_shape
    t_size, c_size, z_size, y_size, x_size = data_shape
    channels = [Channel(id=f"Channel:{i}", name=f"Channel {i}") for i in range(c_size)]
    pixels = Pixels(
        dimension_order="TCZYX",
        type="uint8",
        size_t=t_size,
        size_c=c_size,
        size_z=z_size,
        size_y=y_size,
        size_x=x_size,
        channels=channels,
        tiff_data=[TiffData()]
    )
    image = Image(id="Image:0", name="5D_Image", pixels=pixels)
    ome = OME(images=[image])
    return ome

# Generate OME metadata
ome_metadata = create_ome_metadata(data.shape)
ome_xml = to_xml(ome_metadata)

# Add the structured annotations XML to the OME-XML
structured_annotation = """
<StructuredAnnotations>
    <XMLAnnotation ID="Annotation:3" Namespace="openmicroscopy.org/omero/dimension/modulo">
        <Value>
            <Modulo namespace="http://www.openmicroscopy.org/Schemas/Additions/2011-09">
                <ModuloAlongT Type="lifetime" TypeDescription="TCSPC" Start="0" Step="1" End="9"/>
            </Modulo>
        </Value>
    </XMLAnnotation>
</StructuredAnnotations>
"""

# Insert the structured annotation into the OME-XML
# This should be added just before the closing </OME> tag
ome_xml_with_sa = ome_xml.replace("</OME>", structured_annotation + "</OME>")

# Save the 5D data as an OME-TIFF file with the updated OME-XML metadata
# file_path = '5D_image_with_sa.ome.tif'
# tifffile.imwrite(
#     file_path,
#     data,
#     metadata={"axes": "XYZTC"},
#     description=ome_xml_with_sa,  # Attach the modified OME-XML with structured annotation
# )

file_path = "5D_image_with_sa.zarr"
os.mkdir(file_path)


store = parse_url(file_path, mode="w").store
root = zarr.group(store=store)
write_image(image=data, group=root, axes="tczyx", storage_options=dict(chunks=(5, 1, 5, 64, 64)))


# store = parse_url(file_path, mode="w").store
# root = zarr.group(store=store)
# root.attrs["omero"] = {
#     'metadata': ome_xml_with_sa
# }